# Prerequisite: FAERS ADR Data Prep (FAERS_Prep_SOC_aaai.ipynb)

Content:
1) 
2) limit the outcome in PT to 27K PT that can be mapped to SOC. Since Many PTs are overly broad, such as Condition aggravated, Drug ineffective, Off label use, Product use issue


# SOC related title 
* ontology-Aware Hierarchical Retrieval for Multilabel ADR Prediction
* Hierarchical Pharmacovigilance Retrieval with MedDRA-Guided Evidence Aggregation


In [ ]:
# import tensorflow as tf
# print("TensorFlow version:", tf.__version__)
# print("GPUs available:", tf.config.list_physical_devices('GPU'))

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score, average_precision_score
import pickle, os, sys, time, gc, json, collections
import pandas as pd
import lancedb, lance
from torch.utils.data import DataLoader, Dataset
from lance.vector import vec_to_table
import copy
# from lance.vector import vec_to_table
import numpy as np
import struct
import pyarrow as pab
import pyarrow.dataset
from tqdm import trange, tqdm
import itertools
#from utils.funcs import *
import torch
from torch.utils.data import TensorDataset, DataLoader,random_split
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display, Markdown
#from torchmetrics.text.bert import BERTScore
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import hamming_loss, jaccard_score, recall_score, precision_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"

import warnings
warnings.filterwarnings("ignore")


I0000 00:00:1785431770.531658   76758 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
/home/dada/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a s

# load adr soc training and testing data
* sample data is provided as "adr_trn_soc_spl.pkl" and "adr_tst_soc_spl.pkl"

In [3]:
adr_trn_soc = pickle.load(open("../adr_trn_soc.pkl", "rb"))
adr_tst_soc = pickle.load(open("../adr_tst_soc.pkl", "rb")) 

In [6]:
display(Markdown(f"**Instruction of case id 3593935:** {adr_tst_soc.inst[3593935]}"))

**Instruction of case id 3593935:** {"patient": "{"age": "6.0", "age_cod": "dy", "gndr_cod": "f", "wt": "1.185", "wt_cod": "kg"}", "treatment": "{"drugname": "ibuprofen", "ingredient": "ibuprofen", "atc4": "propionic acid derivatives, other cardiac preparations, antiinflammatory preparations, non-steroids for topical use, antiinflammatory products for vaginal administration, other throat preparations", "route": "intravenous (not otherwise specified)", "dose": "nan nan nan nan"}", "indi_pt": "patent ductus arteriosus", "yr_qtr": "2025q1"}

In [109]:
oot_raw = adr_tst_soc[adr_tst_soc.yr_qtr>="2025Q4"]
print(oot_raw.shape)
oot_raw.head()

(106428, 4)


,caseid,yr_qtr,inst,soc
3760052,258711351,2025Q4,"{""patient"": ""{""age"": ""66.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, ..."
3760053,234073135,2025Q4,"{""patient"": ""{""age"": ""43.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3760054,242482783,2025Q4,"{""patient"": ""{""age"": ""33.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, ..."
3760055,258709621,2025Q4,"{""patient"": ""{""age"": ""55.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
3760056,258714161,2025Q4,"{""patient"": ""{""age"": ""78.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
pickle.dump(oot_raw, open("../inference_oot_soc.pkl", "wb"))

In [ ]:
adr_tst_soc.inst[3593932]

In [111]:
oot_trn = adr_tst_soc[adr_tst_soc.yr_qtr<"2025Q4"]
print(oot_trn.shape)
oot_trn.head(2)

(166120, 4)


,caseid,yr_qtr,inst,soc
3593932,247958291,2025Q1,"{""patient"": ""{""age"": ""76.0"", ""age_cod"": ""yr"", ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3593933,247958231,2025Q1,"{""patient"": ""{""age"": ""50.0"", ""age_cod"": ""yr"", ...","[0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, ..."


In [112]:
pickle.dump(oot_trn, open("../training_oot_soc.pkl", "wb"))

# Evalutate zero shot and few shot SOC predictions

In [7]:
llama_0 = pickle.load(open("soc_llama_0_shot_preds.pkl", "rb"))
llama_0[1]

{'model': './Llama-3.3-70B-Instruct',
 'shots': 0,
 'n_cases': 1000,
 'n_unparsed': 21,
 'macro_precision': 0.21174152481521383,
 'macro_recall': 0.12036829661345667,
 'macro_f1': 0.12802950875271518,
 'micro_precision': 0.256973293768546,
 'micro_recall': 0.13249694002447981,
 'micro_f1': 0.17484352917423784,
 'hamming_loss': 0.15137037037037038,
 'jaccard_samples': 0.10970000416250414}

In [105]:
qwen_0 = pickle.load(open("soc_qwen_0_shot_preds.pkl", "rb"))
qwen_0[1]

{'model': './Qwen2.5-72B-Instruct',
 'shots': 0,
 'n_cases': 1000,
 'n_unparsed': 18,
 'macro_precision': 0.18614103129420367,
 'macro_recall': 0.18604897979625276,
 'macro_f1': 0.16528923554619562,
 'micro_precision': 0.2508871540099361,
 'micro_recall': 0.21634026927784578,
 'micro_f1': 0.23233651002300362,
 'hamming_loss': 0.17303703703703704,
 'jaccard_samples': 0.1355102046102046}

In [90]:
llama_3 = pickle.load(open("soc_llama_3_shot_preds.pkl", "rb"))
llama_3[1]

{'model': './Llama-3.3-70B-Instruct',
 'shots': 3,
 'n_cases': 1000,
 'n_unparsed': 42,
 'macro_precision': 0.22976885427955457,
 'macro_recall': 0.2389067044603277,
 'macro_f1': 0.20820922024072297,
 'micro_precision': 0.27151051625239003,
 'micro_recall': 0.2810291207237772,
 'micro_f1': 0.27618782995276464,
 'hamming_loss': 0.19296296296296298,
 'jaccard_samples': 0.15091910981175688}

In [91]:
qwen_3 = pickle.load(open("soc_qwen_3_shot_preds.pkl", "rb"))
qwen_3[1]

{'model': './Qwen2.5-72B-Instruct',
 'shots': 3,
 'n_cases': 1000,
 'n_unparsed': 2,
 'macro_precision': 0.21652745016670416,
 'macro_recall': 0.24229251876221783,
 'macro_f1': 0.19845011756997816,
 'micro_precision': 0.2530149541726966,
 'micro_recall': 0.29657902176986145,
 'micro_f1': 0.2730704152023949,
 'hamming_loss': 0.20685185185185184,
 'jaccard_samples': 0.15426475027380598}

In [ ]:
qwen_c = pickle.load(open("soc_qwen_context_long_preds.pkl", "rb"))
qwen_c[1]

In [ ]:
llama_c = pickle.load(open("soc_llama_context_long_preds.pkl", "rb"))
llama_c[1] #new

# Combine llama_c, qwen_c, llama_3, qwen_3, llama_0, and qwen_0

In [99]:
llm_out = pd.DataFrame()
for dict in (llama_c[1], qwen_c[1], llama_3[1], qwen_3[1], llama_0[1], qwen_0[1]):
    llm_out = pd.concat([llm_out, pd.DataFrame([dict])])

In [100]:
llm_out["Inference"] = ["Llama_Context", "Qwen_Context", "Llama_3_Shots", "Qwen_3_Shots",
                        "Llama_Zero_shot","Qwen_Zero_shot"] 

In [101]:
llm_tbl = llm_out[["Inference", "macro_precision","macro_recall","macro_f1","micro_precision",
        "micro_recall","micro_f1","hamming_loss","jaccard_samples"]]

In [102]:
llm_tbl

,Inference,macro_precision,macro_recall,macro_f1,micro_precision,micro_recall,micro_f1,hamming_loss,jaccard_samples
0,Llama_Context,0.332978,0.355841,0.320302,0.389973,0.497793,0.437336,0.150481,0.304567
0,Qwen_Context,0.338444,0.364963,0.315032,0.365469,0.497163,0.421264,0.160481,0.290886
0,Llama_3_Shots,0.229769,0.238907,0.208209,0.271511,0.281029,0.276188,0.192963,0.150919
0,Qwen_3_Shots,0.216527,0.242293,0.198450,0.253015,0.296579,0.273070,0.206852,0.154265
0,Llama_Zero_shot,0.211742,0.120368,0.128030,0.256973,0.132497,0.174844,0.151370,0.109700
0,Qwen_Zero_shot,0.186141,0.186049,0.165289,0.250887,0.216340,0.232337,0.173037,0.135510


# output latex table

In [103]:
print(llm_tbl.iloc[:,:4].to_latex(index=False,
                  formatters={"name": str.upper},
                  float_format="{:.3f}".format,
                  ))

\begin{tabular}{lrrr}
\toprule
Inference & macro_precision & macro_recall & macro_f1 \\
\midrule
Llama_Context & 0.333 & 0.356 & 0.320 \\
Qwen_Context & 0.338 & 0.365 & 0.315 \\
Llama_3_Shots & 0.230 & 0.239 & 0.208 \\
Qwen_3_Shots & 0.217 & 0.242 & 0.198 \\
Llama_Zero_shot & 0.212 & 0.120 & 0.128 \\
Qwen_Zero_shot & 0.186 & 0.186 & 0.165 \\
\bottomrule
\end{tabular}



In [ ]:
pickle.dump(llm_tbl, open("llm_inf_tbl.pkl", "wb"))
#llm_tbl = pickle.load(open("llm_inf_tbl.pkl", "rb"))

# load full version of training and test df

In [ ]:
# # load adr_trn and adr_tst
# # adr_trn, adr_tst = pickle.load(open("adr_trn_tst.pkl", "rb"))
# adr_all = pickle.load(open("../adr_all_new_up2_26q1.pkl", "rb"))

In [ ]:
adr_trn_soc.yr_qtr.max()

In [ ]:
adr_tst_soc.yr_qtr.value_counts()

In [ ]:
# 3 quarter to training and 2 quarter for testing
adr_trn = adr_tst_soc[adr_tst_soc.yr_qtr <= '2025Q3']
adr_tst = adr_tst_soc[adr_tst_soc.yr_qtr > '2025Q3']

In [ ]:
adr_tst_soc.head()

In [ ]:
adr_tst_soc.inst[3593936]

# loading hybrid retrieval ids and scores


In [ ]:
(vect_ids, vect_scores) = pickle.load(open("soc_tst_vect_id_score_lst.pkl", "rb"))
(bm25_ids, bm25_scores) = pickle.load(open("../bm25_soc_oot_ids_scores.pkl", "rb"))


In [ ]:
bm25_scores[0]

In [ ]:
len(vect_ids)  == len(adr_tst_soc)

In [ ]:
def normalize_scores(scores):
    mean = np.mean(scores)
    std = np.std(scores)
    return (scores - mean) / std

In [ ]:
normalize_scores(vect_scores[0])

In [ ]:
# normalized vect_scores and bm25_scores using the mean and std from the training set
vect_scores = [normalize_scores(scores) for scores in vect_scores]
bm25_scores = [normalize_scores(scores) for scores in bm25_scores]

In [ ]:
vect_ids = [[int(j) for j in j_lst] for j_lst in vect_ids]

In [ ]:
vect_ids[0]

In [ ]:
print(len(vect_ids))
len(vect_ids) == len(bm25_ids)

In [ ]:
# Save id and score back
pickle.dump((vect_ids, vect_scores), open("soc_tst_vect_id_score_lst.pkl", "wb"))
pickle.dump((bm25_ids, bm25_scores), open("../bm25_soc_oot_ids_scores.pkl", "wb"))

In [ ]:
(bm25_ids, bm25_scores) = pickle.load(open("../bm25_soc_oot_ids_scores.pkl", "rb"))

In [ ]:
bm25_scores[0]

In [ ]:
# based on the id, get multilabels
print(adr_trn_soc.shape)
adr_trn_soc.head()

In [ ]:
# get soc labels of vect_ids from adr_trn_soc
bm25_lbl = []
vect_lbl = []

for i in trange(len(vect_ids)):
    bm25_i = bm25_ids[i]
    bm25_lbl.append(adr_trn_soc.soc[bm25_i].tolist())

    vect_i = vect_ids[i]
    vect_lbl.append(adr_trn_soc.soc[vect_i].tolist())  

In [ ]:
pickle.dump((vect_lbl, bm25_lbl), open("bm25_vect_llbs.pkl", "wb"))

# split TST set into training and testing
* use yr_qtr of 2025q1-q3 for training 
* testing on "2025q4" and "2026q1"

In [ ]:
trn_idx = adr_trn.index.tolist()
tst_idx = adr_tst.index.tolist()
len(tst_idx)


In [ ]:
#pickle.dump((trn_idx, tst_idx), open("trn_val_idx.pkl", "wb"))
trn_idx, tst_idx = pickle.load(open("trn_val_idx.pkl", "rb"))

In [ ]:
len(trn_idx)

############################################
# Joint PT-SOC Model w/ Multi-Head Attention


In [ ]:
# load adr inst embedding from ARC
adr_all_embed = pickle.load(open("../adr_inst_embeds.pkl", "rb"))

In [ ]:
adr_all_embed.shape

In [ ]:
# Build query embeddings using the same row-id mapping as the retrieval lists.
row_id_to_pos = {int(row_id): pos for pos, row_id in enumerate(adr_tst_soc.index.tolist())}
train_row_ids = [int(row_id) for row_id in adr_trn.index.tolist()]
test_row_ids = [int(row_id) for row_id in adr_tst.index.tolist()]


def build_embed_subset(row_ids, embed_array):
    if len(embed_array) == 0:
        raise ValueError("Embedding array is empty")

    zero_vec = np.zeros_like(np.asarray(embed_array[0], dtype=np.float32))
    out = []
    for row_id in row_ids:
        pos = row_id_to_pos.get(int(row_id))
        if pos is None:
            out.append(zero_vec.copy())
            continue
        if 0 <= pos < len(embed_array):
            out.append(np.asarray(embed_array[pos], dtype=np.float32))
        else:
            out.append(zero_vec.copy())
    return np.stack(out, axis=0)


adr_trn_embed = build_embed_subset(train_row_ids, adr_all_embed)
adr_tst_embed = build_embed_subset(test_row_ids, adr_all_embed)

pickle.dump(adr_trn_embed, open("adr_soc_trn_embeds.pkl", "wb"))
pickle.dump(adr_tst_embed, open("adr_soc_tst_embeds.pkl", "wb"))

In [ ]:
adr_trn_embed = pickle.load(open("adr_soc_trn_embeds.pkl", "rb")) # 2 quarters

In [ ]:
adr_tst_embed = pickle.load(open("adr_soc_tst_embeds.pkl", "rb"))

In [ ]:
# map id to embed for quick lookup
adr_trn_embed = dict(zip(adr_trn.index.tolist(), adr_trn_embed)) 
adr_tst_embed = dict(zip(adr_tst.index.tolist(), adr_tst_embed))  

In [ ]:
# get top5 pt embeds for bm25 and vect pipelines based on id list
full_row_ids = [int(x) for x in adr_tst_soc.index.tolist()]
row_pos = {int(row_id): pos for pos, row_id in enumerate(full_row_ids)}


def build_retrieval_embeds(retrieval_lists, split_ids, embed_array):
    if len(embed_array) == 0:
        raise ValueError("Embedding array is empty")

    zero_vec = np.zeros_like(np.asarray(embed_array[0], dtype=np.float32))
    out = []

    for row_id in split_ids:
        row_pos_idx = row_pos.get(int(row_id))
        if row_pos_idx is None:
            out.append([zero_vec.copy() for _ in range(5)])
            continue

        ids_for_row = retrieval_lists[row_pos_idx]
        row_embeds = []
        for cand_id in ids_for_row:
            cand_id = int(cand_id)
            if 0 <= cand_id < len(embed_array):
                row_embeds.append(np.asarray(embed_array[cand_id], dtype=np.float32))
            else:
                row_embeds.append(zero_vec.copy())

            if len(row_embeds) >= 5:
                break

        if len(row_embeds) < 5:
            row_embeds += [zero_vec.copy() for _ in range(5 - len(row_embeds))]

        out.append(row_embeds[:5])

    return out

trn_bm25_pt_embeds = build_retrieval_embeds(bm25_ids, trn_idx, adr_all_embed)
trn_vect_pt_embeds = build_retrieval_embeds(vect_ids, trn_idx, adr_all_embed)
tst_bm25_pt_embeds = build_retrieval_embeds(bm25_ids, tst_idx, adr_all_embed)
tst_vect_pt_embeds = build_retrieval_embeds(vect_ids, tst_idx, adr_all_embed)

bm25_pt_embeds = trn_bm25_pt_embeds
vect_pt_embeds = trn_vect_pt_embeds


In [ ]:
del adr_all_embed #clear memory
gc.collect()

In [ ]:
pickle.dump((trn_bm25_pt_embeds, trn_vect_pt_embeds), open("bm25_vect_top5_soc_embeds.pkl", "wb"))
bm25_pt_embeds = trn_bm25_pt_embeds
vect_pt_embeds = trn_vect_pt_embeds


In [ ]:
(trn_bm25_pt_embeds, trn_vect_pt_embeds) = pickle.load(open("bm25_vect_top5_soc_embeds.pkl", "rb"))


In [ ]:
len(trn_bm25_pt_embeds[0][0])

In [ ]:
# Map each training row to the SOC index used by the model.
# This uses the SOC labels from adr_trn_soc and the retrieved PT IDs from vect_ids.

if 'adr_trn_soc' in globals() and 'vect_ids' in globals():
    soc_labels = adr_trn_soc['soc'].astype(str).tolist()
    soc_to_idx = {soc: i for i, soc in enumerate(sorted(set(soc_labels)))}
    trn_soc_idx = np.array([soc_to_idx.get(soc, -1) for soc in soc_labels], dtype=np.int64)
    print(f"SOC label count: {len(soc_to_idx)}")
    print(f"Training SOC index shape: {trn_soc_idx.shape}")
else:
    print("adr_trn_soc or vect_ids is not available in the current notebook state.")


# Prep training target: y_trn


In [ ]:
# print(adr_tst_soc.shape)
# adr_tst_soc.head()

In [ ]:
# target 
y_trn_1hot = torch.tensor(adr_tst_soc.loc[adr_tst_soc.yr_qtr.isin(["2025Q1", "2025Q2","2025Q3"]), "soc"].tolist(), 
                      dtype=torch.int32)  # (27,)
y_trn_1hot.shape

In [ ]:
# target 
y_oot_1hot = torch.tensor(adr_tst_soc.loc[adr_tst_soc.yr_qtr.isin(["2025Q4", "2026Q1"]), "soc"].tolist(), 
                      dtype=torch.int32)  # (27,)
y_oot_1hot.shape


In [ ]:
#pickle.dump((y_trn_1hot,y_oot_1hot), open("y_trn_oot.pkl", "wb"))
(y_trn_1hot,y_oot_1hot) = pickle.load(open("y_trn_oot.pkl", "rb"))

# Define SOC multi-hot model, prior, and loss function


In [ ]:
num_soc_classes = 27 #len(soc_classes)
y_train_mh = y_trn_1hot.float()
soc_pos_rate, soc_prior_logits = compute_soc_priors(y_train_mh)

pos_counts = y_train_mh.sum(dim=0)
neg_counts = y_train_mh.size(0) - pos_counts
soc_pos_weight = torch.sqrt(neg_counts / pos_counts.clamp(min=1.0)).to(device)


In [ ]:
import sys
from pathlib import Path
import importlib


import importlib
import util.multihot_loss_comparison as my_module

# After changing the file...
importlib.reload(my_module)
from util.multihot_loss_comparison import *


# Prepare rich_token data for attention model

In [ ]:
# --- aligned component arrays (272548 = 166120 train ++ 106428 oot) ---
_bm_lab, _dn_lab = pickle.load(open("bm25_vect_llbs.pkl", "rb"))               # labels
_, _dn_sc = pickle.load(open("soc_tst_vect_id_score_lst.pkl", "rb"))           # dense scores
_, _bm_sc = pickle.load(open("../bm25_soc_oot_ids_scores.pkl", "rb"))          # bm25 scores 
_be0, _be1 = pickle.load(open("bm25_vect_top5_soc_embeds.pkl", "rb"))          # bm25, dense embeds
_ytr, _yte = pickle.load(open("y_trn_oot.pkl", "rb"))

In [ ]:
_dn_sc = np.asarray([normalizeVect(x_dn_sc) for x_dn_sc in _dn_sc])
_bm_sc = np.asarray([normalizeVect(x_bm_sc) for x_bm_sc in _bm_sc])

In [ ]:
lab = np.concatenate([np.asarray(_bm_lab, np.float16), np.asarray(_dn_lab, np.float16)], 1)  # (N,10,27)
sc  = np.concatenate([np.asarray(_bm_sc,  np.float16), np.asarray(_dn_sc,  np.float16)], 1)  # (N,10)
emb = np.concatenate([np.asarray(_be0,    np.float16), np.asarray(_be1,    np.float16)], 1)  # (N,10,1024)
y_all = torch.cat([_ytr.float(), _yte.float()])                                              # (N,27)
del _bm_lab, _dn_lab, _be0, _be1; gc.collect()
N = lab.shape[0]; assert N == 272548, N

In [ ]:
y_trn = _ytr.numpy()
y_trn.shape

In [ ]:
X_tst = X16.numpy()[166120:]; y_tst = _yte.numpy()

In [ ]:
y_tst.shape

In [ ]:
pickle.dump((X_trn.reshape(X_trn.shape[0],1,-1), X_tst.reshape(X_tst.shape[0],1,-1), y_trn, y_tst), \
            open("ML_trn_tst.pkl", "wb"))

In [ ]:
tt = adr_tst_soc[adr_tst_soc.yr_qtr == "2025Q4"]

In [ ]:
tt.shape

In [ ]:
#normalize the scores in the rich tensor for each of the 10 candidates per sample
X16.numpy()[0][:, 0]

In [ ]:
#load ML trn and tst data
(X_trn, X_tst, y_trn, y_tst) = pickle.load(open("ML_trn_tst.pkl", "rb"))

In [ ]:
del X_trn; gc.collect()

# fit baseline model - XGBoost

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [ ]:
# # Option A: Logistic Regression (Fast and baseline standard)
# base_lr = LogisticRegression(max_iter=1000)
# ml_model = MultiOutputClassifier(base_lr)
# ml_model.fit(X_trn.reshape(X_trn.shape[0], -1), y_trn)  # X_train shape: (num_samples, vector_dim)
#                                 # y_train shape: (num_samples, 27)



In [ ]:
%%time 

# Option B: LightGBM (Better accuracy for non-linear vector spaces)
base_lgb = XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=6, objective='binary:logistic', device='cuda')
ml_model_lgb = MultiOutputClassifier(base_lgb)
ml_model_lgb.fit(X_trn.reshape(X_trn.shape[0], -1), y_trn)

In [ ]:
#pickle.dump(ml_model_lgb, open("ml_model_xgboost.pkl", "wb"))
ml_model_lgb = pickle.load(open("ml_model_xgboost.pkl", "rb"))

In [ ]:
ml_cls = ml_model_lgb.predict(X_tst.reshape(X_tst.shape[0], -1))
ml_prob = ml_model_lgb.predict_proba(X_tst.reshape(X_tst.shape[0], -1))

In [ ]:
pickle.dump((ml_cls, ml_prob), open("ml_cls_prob_xgboost.pkl", "wb"))

In [ ]:
# integrate ML predictions with retrieval scores for final SOC prediction

In [ ]:
from util.multihot_loss_comparison import (
    _macro_metrics, macro_roc_auc, macro_average_precision, KnnVoteResult, _KNN_VOTE_DEFAULT_VARIANTS,
)

# ml_prob is a list of (n,2) arrays (one per SOC, from MultiOutputClassifier) -> (N,27) pos-class prob
ml_prob_mat = np.stack(
    [p[:, 1] if p.shape[1] > 1 else np.zeros(len(p), dtype=np.float32) for p in ml_prob],
    axis=1,
)

ml_p, ml_r, ml_f1, ml_density = _macro_metrics(y_tst, ml_cls)
ml_auroc = macro_roc_auc(y_tst, ml_prob_mat)
ml_auprc = macro_average_precision(y_tst, ml_prob_mat)

ml_result = KnnVoteResult(
    variant_name="ML classifier (XGBoost)",
    threshold_mode="predict>=0.5",
    val_macro_p=ml_p,
    val_macro_r=ml_r,
    val_macro_f1=ml_f1,
    val_macro_auroc=ml_auroc,
    val_macro_auprc=ml_auprc,
    val_pred_density=ml_density,
)
print(f"ML classifier (XGBoost):  macroF1={ml_f1:.4f} P={ml_p:.4f} R={ml_r:.4f} "
      f"AUPRC={ml_auprc:.4f} AUROC={ml_auroc:.4f} density={ml_density:.3f}")

In [ ]:
pickle.dump(ml_result, open("ml_result.pkl", "wb"))

In [ ]:
seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)

dataset_rich = TensorDataset(torch.tensor(X16.numpy()[:166120]), _ytr)

In [ ]:
# 5-fold CV
n = len(dataset_rich)
train_size = int(0.8 * n)
val_size = n - train_size
train_rich, val_rich = random_split(dataset_rich, [train_size, val_size])

#set batch size of 2048 to use GPU
rich_train_loader = DataLoader(train_rich, batch_size=2048, shuffle=True, num_workers=2, pin_memory=True)
rich_val_loader = DataLoader(val_rich, batch_size=2048, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
oot_rich = TensorDataset(torch.tensor(X16.numpy()[166120:]), _yte)
rich_oot_loader = DataLoader(oot_rich, batch_size=2048, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
X16.numpy().shape

In [ ]:
#52012
oot_rich_2025q4 = TensorDataset(torch.tensor(X16.numpy()[166120:166120+52012]), _yte[:52012])
oot_rich_2026q1 = TensorDataset(torch.tensor(X16.numpy()[166120+52012:]), _yte[52012:])
rich_oot_2025q4_loader = DataLoader(oot_rich_2025q4, batch_size=2048, shuffle=False, num_workers=2, pin_memory=True)
rich_oot_2026q1_loader = DataLoader(oot_rich_2026q1, batch_size=2048, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# pickle.dump((rich_train_loader, rich_val_loader, rich_oot_loader), 
#             open("normalized_rich_trn_val_oot_loader.pkl", "wb"))
#(rich_train_loader, rich_val_loader, rich_oot_loader) = pickle.load(open("normalized_rich_trn_val_oot_loader.pkl", "rb"))
(rich_train_loader, rich_val_loader, rich_oot_loader) = pickle.load(open("normalized_rich_trn_val_oot_loader.pkl", "rb"))

In [ ]:
oot_set = rich_oot_loader.dataset

In [ ]:
y_tr = _ytr
_pr = (y_tr.sum(0) / y_tr.shape[0]).clamp(1e-4, 1 - 1e-4)
rich_prior_logits = torch.log(_pr / (1 - _pr)).to(device)
_pos = y_tr.sum(0); _neg = y_tr.shape[0] - _pos
rich_pos_weight = torch.sqrt(_neg / _pos.clamp(min=1.0)).to(device)

# visualize model architecture

In [ ]:
num_soc_classes = 27 # 27 unique level of SOC

In [ ]:
# SOCLabelQueryModel is defined in utility file
model = SOCLabelQueryModel(
        token_dim=1052, 
        hidden=768, 
        num_soc_classes=num_soc_classes,
        dropout=0.2, 
        prior_logits=rich_prior_logits, 
        num_heads=4,
        ).to(device)

In [ ]:
plot_soc_label_query_model_diagram(model, batch_size = 1)

In [ ]:
#visualize_soc_label_query_model(model, batch_size = 1)

In [ ]:
# print model architecture
print_soc_label_query_model_structure(model)

In [ ]:
# Store results
scan_results = []

# Early stopping setup
val_patience = 5
num_soc_classes = 27
epochs_per_hparams = 100
seeds = (1234, 42, 7)

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Create criterion with these hyperparams
    criterion =  nn.BCEWithLogitsLoss().to(device)

    # define _rich_labelquery_factory
    # 1 retrieval score + 27 soc labels + 1024 embedding
    model = SOCLabelQueryModel(
        token_dim=1052, 
        hidden=768, 
        num_soc_classes=num_soc_classes,
        dropout=0.2, 
        prior_logits=rich_prior_logits, 
        num_heads=4,
        ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

    # Mini training loop
    best_val_f1 = -1.0
    epochs_no_improve = 0
    best_state = None
    best_epoch = 0

    for epoch in trange(epochs_per_hparams):
        model.train()
        total_loss = 0.0
        n_train = 0
        
        #, rich_val_loader,train_loader_1hot:
        for x_batch, y_batch in rich_train_loader: 
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device).float()
            
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item() * x_batch.size(0)
            n_train += x_batch.size(0)
        
        train_loss = total_loss / max(n_train, 1)
        
        # Validation
        va_loss, va_f1, va_rec, pred_rate = eval_multihot_loader(
            model, rich_val_loader, criterion, threshold=0.35
        )
        
        if va_f1 > best_val_f1 + 1e-4:
            best_val_f1 = va_f1
            best_epoch = epoch + 1
            epochs_no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= val_patience:
                break

    # Load best state
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)

    # Final eval on best model tuned on F1
    y_val, probs_val = collect_multihot_probs(model, rich_val_loader) #val_loader_1hot)
    val_threshold, f1 = tune_multihot_threshold_f1(y_val, probs_val)

    y_pred = (probs_val >= val_threshold).astype(float)
    final_p = precision_score(y_val, y_pred, average="macro", zero_division=0)
    final_r = recall_score(y_val, y_pred, average="macro", zero_division=0)
    final_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
    final_hl = hamming_loss(y_val, y_pred)
    final_jc = jaccard_score(y_val, y_pred, average="samples", zero_division=0)
    final_auc = roc_auc_score(y_val, y_pred, average="macro")
    final_pr_auc = average_precision_score(y_val, y_pred, average="macro")
    density = y_pred.mean()

    scan_results.append({
        'best_epoch': best_epoch,
        'val_threshold': val_threshold,
        'val_macro_f1': final_f1,
        'val_macro_p': final_p,
        'val_macro_r': final_r,
        "val_auc": final_auc,
        "val_pr_auc": final_pr_auc,
        'val_hamming_loss': final_hl,
        'val_jaccard_score': final_jc,
        #'train_loss_final': train_loss,
        'density': density,
    })    

    # save model to disk
    torch.save(model.state_dict(), f"rich_token_seed_{seed}.pt")


In [ ]:
# Create results dataframe and display - tuned by f1
scan_df = pd.DataFrame(scan_results)
scan_df_sorted = scan_df.sort_values('val_macro_f1', ascending=False)
scan_df.to_csv("normalized_rich_token_by_seed_final.csv", index=False)
print(scan_df_sorted)

In [ ]:
# Create results dataframe and display - tuned by f1
scan_df = pd.read_csv("rich_token_by_seed_final.csv")
scan_df_sorted = scan_df.sort_values('val_macro_f1', ascending=False)
print(scan_df_sorted)

In [ ]:
# Create results dataframe and display - tuned by recall
scan_df = pd.DataFrame(scan_results)
scan_df_sorted = scan_df.sort_values('val_macro_f1', ascending=False)
scan_df.to_csv("rich_token_by_seed_recall.csv", index=False)
print(scan_df_sorted)

# load utilitis func

In [ ]:
import importlib
import util.multihot_loss_comparison as my_module

# After changing the file...
importlib.reload(my_module)
from util.multihot_loss_comparison import *


# eval 3 model performance on oot set
* rich_oot_loader
* model by seed: rich_token_seed_1234.pt, rich_token_seed_42.pt, rich_token_seed_7.pt

In [ ]:
seeds = (1234, 42, 7)

In [ ]:
ckpt_paths = [f"rich_token_seed_{seed}.pt" for seed in seeds]
thresholds = [0.3, 0.3, 0.3]

In [ ]:
ckpt_dict = dict(zip(ckpt_paths, thresholds))
ckpt_dict

In [ ]:
def _rich_labelquery_factory():
    return SOCLabelQueryModel(
        token_dim=1052, hidden=768, num_soc_classes=num_soc_classes,
        dropout=0.2, prior_logits=rich_prior_logits, num_heads=4,
    )

In [ ]:
# Create one checkpoint per seed from the trained rich-token runs and evaluate on the OOT loader.
oot_rows_2025q4 = []
oot_rows_2026q1 = []
val_rows = []
for ckpt_path , threshold in ckpt_dict.items():
    model = _rich_labelquery_factory().to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    # Evaluate on OOT loader
    y_true, probs = collect_multihot_probs(model, rich_oot_2025q4_loader)
    pred = (probs >= threshold).astype(float)
    prec_macro = precision_score(y_true, pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_true, pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, pred, average="macro", zero_division=0)
    auprc_macro = average_precision_score(y_true, probs, average="macro") 
    prec_micro = precision_score(y_true, pred, average="micro", zero_division=0)
    rec_micro = recall_score(y_true, pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, pred, average="micro", zero_division=0)
    auprc_micro = average_precision_score(y_true, probs, average="micro")
    oot_rows_2025q4.append({
        #"seed": seed,
        "checkpoint": str(ckpt_path),
        "threshold": threshold,
        "macro_precision": prec_macro,
        "macro_recall": rec_macro,
        "macro_f1": f1_macro,
        "macro_auprc": auprc_macro,
        "micro_precision": prec_micro,
        "micro_recall": rec_micro,
        "micro_f1": f1_micro,
        "micro_auprc": auprc_micro,
    })
    
    y_true, probs = collect_multihot_probs(model, rich_oot_2026q1_loader)
    pred = (probs >= threshold).astype(float)
    prec_macro = precision_score(y_true, pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_true, pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, pred, average="macro", zero_division=0)
    auprc_macro = average_precision_score(y_true, probs, average="macro") 
    prec_micro = precision_score(y_true, pred, average="micro", zero_division=0)
    rec_micro = recall_score(y_true, pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, pred, average="micro", zero_division=0)
    auprc_micro = average_precision_score(y_true, probs, average="micro")
    oot_rows_2026q1.append({
        #"seed": seed,
        "checkpoint": str(ckpt_path),
        "threshold": threshold,
        "macro_precision": prec_macro,
        "macro_recall": rec_macro,
        "macro_f1": f1_macro,
        "macro_auprc": auprc_macro,
        "micro_precision": prec_micro,
        "micro_recall": rec_micro,
        "micro_f1": f1_micro,
        "micro_auprc": auprc_micro,
    })

    # evaluate on validation loader
    y_val, probs_val = collect_multihot_probs(model, rich_val_loader)
    pred_val = (probs_val >= threshold).astype(float)
    prec_macro = precision_score(y_val, pred_val, average="macro", zero_division=0)
    rec_macro = recall_score(y_val, pred_val, average="macro", zero_division=0)
    f1_macro = f1_score(y_val, pred_val, average="macro", zero_division=0)
    auprc_macro = average_precision_score(y_val, probs_val, average="macro") 
    prec_micro = precision_score(y_val, pred_val, average="micro", zero_division=0)
    rec_micro = recall_score(y_val, pred_val, average="micro", zero_division=0)
    f1_micro = f1_score(y_val, pred_val, average="micro", zero_division=0)
    auprc_micro = average_precision_score(y_val, probs_val, average="micro")
    val_rows.append({
        #"seed": seed,
        "checkpoint": str(ckpt_path),
        "threshold": threshold,
        "macro_precision": prec_macro,
        "macro_recall": rec_macro,
        "macro_auprc": auprc_macro,
        "macro_f1": f1_macro,
        "micro_precision": prec_micro,
        "micro_recall": rec_micro,
        "micro_f1": f1_micro,
        "micro_auprc": auprc_micro,
    })

In [ ]:
# oot_rows_2026q1_df = pd.DataFrame(oot_rows_2026q1).sort_values("macro_f1", ascending=False)
# print("\nOOT evaluation for rich-token checkpoints:")
# display(oot_rows_2026q1_df)
# oot_rows_2026q1_df.to_csv("./loss_experiments/outputs/rich_token_oot_2026q1_eval.csv", index=False)
oot_rows_2026q1_df = pd.read_csv("./loss_experiments/outputs/rich_token_oot_2026q1_eval.csv")

In [ ]:
# oot_rows_2025q4_df = pd.DataFrame(oot_rows_2025q4).sort_values("macro_f1", ascending=False)
# print("\nOOT evaluation for rich-token checkpoints:")
# display(oot_rows_2025q4_df)
# oot_rows_2025q4_df.to_csv("./loss_experiments/outputs/rich_token_oot_2025q4_eval.csv", index=False)
oot_rows_2025q4_df = pd.read_csv("./loss_experiments/outputs/rich_token_oot_2025q4_eval.csv")

In [ ]:
# val_eval_df = pd.DataFrame(val_rows).sort_values("macro_f1", ascending=False)
# print("\nValidation evaluation for rich-token checkpoints:")
# display(val_eval_df)
# val_eval_df.to_csv("./loss_experiments/outputs/rich_token_val_eval.csv", index=False)
val_eval_df = pd.read_csv("./loss_experiments/outputs/rich_token_val_eval.csv")

In [ ]:
import pickle, gc, numpy as np, torch, importlib
import loss_experiments.multihot_loss_comparison as my_module
importlib.reload(my_module)
from loss_experiments.multihot_loss_comparison import *

In [ ]:
summary_val_oot_experiment(val_eval_df, oot_rows_2025q4_df, oot_rows_2026q1_df, width = 0.22)

# kNN Lable-Vote Baseline 
* correctly aligned neighbor labels + real scores

In [ ]:
# ============================================================================
# labels : bm25_vect_llbs.pkl  -> (BM25 5x27, dense 5x27) per query
# dense scores : soc_tst_vect_id_score_lst.pkl[1]   (cosine-ish, ~[-2,2])
# BM25 scores  : ../bm25_soc_oot_ids_scores.pkl[1]  (raw BM25, ~[6,1.4e4]; NaN fixed upstream)
# targets : y_trn_oot.pkl  (train 166120 ++ test 106428) — SAME split as soc_trn_X
# Q: does score-weighting help the vote?
#   -> unweighted all-10 mean wins on macro-F1; weighting only helps AUPRC (ranking).
# BM25 and dense scores live on very different scales -> per-source normalization.
# Threshold: default_mode="f1_per_class" (per-SOC F1-max threshold, tuned on train),
# matching train_one_loss's mis-thresholding fix; results + plot mirror
# summary_comparison_experiment() via summary_knn_vote_experiment().
# ============================================================================
# import pickle, numpy as np, torch
# from loss_experiments.multihot_loss_comparison import (
#     evaluate_knn_vote, summary_knn_vote_experiment,
# )

_bm25_lab, _dense_lab = pickle.load(open("bm25_vect_llbs.pkl", "rb"))
_, _dense_score = pickle.load(open("soc_tst_vect_id_score_lst.pkl", "rb"))
_, _bm25_score  = pickle.load(open("../bm25_soc_oot_ids_scores.pkl", "rb"))

# load y_trn for Q1-Q3'25
_ytr, _yte = pickle.load(open("y_trn_oot.pkl", "rb"))
y_knn = torch.cat([_ytr.float(), _yte.float()]).numpy()

bm25_lab  = np.asarray(_bm25_lab,  dtype=np.float32)          # (N,5,27)
dense_lab = np.asarray(_dense_lab, dtype=np.float32)
bm25_sc   = np.asarray(_bm25_score,  dtype=np.float32)        # (N,5) NaN fixed upstream
dense_sc  = np.asarray(_dense_score, dtype=np.float32)

# # --- aligned component arrays (272548 = 166120 train ++ 106428 oot) ---
# _bm_lab, _dn_lab = pickle.load(open("bm25_vect_llbs.pkl", "rb"))               # labels
# _, _dn_sc = pickle.load(open("soc_tst_vect_id_score_lst.pkl", "rb"))           # dense scores
# _, _bm_sc = pickle.load(open("../bm25_soc_oot_ids_scores.pkl", "rb"))          # bm25 scores 
# _be0, _be1 = pickle.load(open("bm25_vect_top5_soc_embeds.pkl", "rb"))          # bm25, dense embeds
# _ytr, _yte = pickle.load(open("y_trn_oot.pkl", "rb"))

# lab = np.concatenate([np.asarray(_bm_lab, np.float16), np.asarray(_dn_lab, np.float16)], 1)  # (N,10,27)
# sc  = np.concatenate([np.asarray(_bm_sc,  np.float16), np.asarray(_dn_sc,  np.float16)], 1)  # (N,10)
# emb = np.concatenate([np.asarray(_be0,    np.float16), np.asarray(_be1,    np.float16)], 1)  # (N,10,1024)
# y_all = torch.cat([_ytr.float(), _yte.float()])                                              # (N,27)
# del _bm_lab, _dn_lab, _be0, _be1; gc.collect()
# N = lab.shape[0]; assert N == 272548, N


N = len(y_knn)
knn_tr, knn_te = np.arange(0, N), np.arange(166120, N)   # native train/test

GRID = np.arange(0.05, 1.0, 0.05)                             # coarse grid ~1 min on CPU
print(f"aligned kNN data: N={N} labels{bm25_lab.shape} | true density {y_knn.mean():.4f} "
      f"| bm25 score [{bm25_sc.min():.1f},{bm25_sc.max():.1f}] dense [{dense_sc.min():.2f},{dense_sc.max():.2f}]")

def _norm_w(s):
    s = np.clip(s, 0, None); ss = s.sum(1, keepdims=True)
    return np.divide(s, ss, out=np.full_like(s, 1.0 / s.shape[1]), where=ss > 0)

def _vote(prior, name, threshold_mode="f1_per_class"):
    res = evaluate_knn_vote(
        y_knn, prior, knn_tr, knn_te, name,
        threshold_mode=threshold_mode, thresholds=GRID,
    )
    print(f"  {res.variant_name:34s} AUPRC={res.val_macro_auprc:.4f} AUROC={res.val_macro_auroc:.4f} "
          f"macroF1={res.val_macro_f1:.4f} P={res.val_macro_p:.4f} R={res.val_macro_r:.4f} "
          f"dens={res.val_pred_density:.3f}")
    return res

all_lab = np.concatenate([bm25_lab, dense_lab], 1)                        # (N,10,27)
all_w   = np.concatenate([_norm_w(bm25_sc), _norm_w(dense_sc)], 1) / 2.0  # per-source normalized

knn_vote_results = []
print("\n-- NO score weight (plain mean) --")
knn_vote_results.append(_vote(dense_lab.mean(1), "dense mean (5)"))
knn_vote_results.append(_vote(bm25_lab.mean(1),  "bm25 mean (5)"))
knn_vote_results.append(_vote(all_lab.mean(1),   "all 10 mean"))

print("-- WITH score weight (fixed bm25 scores, per-source normalized) --")
knn_vote_results.append(_vote((_norm_w(dense_sc)[:, :, None] * dense_lab).sum(1), "dense score-weighted"))
knn_vote_results.append(_vote((_norm_w(bm25_sc)[:, :, None] * bm25_lab).sum(1),   "bm25 score-weighted"))
knn_vote_results.append(_vote((all_w[:, :, None] * all_lab).sum(1),               "all 10 score-weighted"))


In [ ]:
pickle.dump(knn_vote_results, open("knn_vote_results.pkl", "wb"))

In [ ]:
# Ranked table + grouped-bar plot (macro F1/Recall/Precision/AUPRC), same shape as
# summary_comparison_experiment()'s output for the trained-model comparison, with the
# ML classifier (XGBoost) folded in alongside the kNN vote variants.
knn_vote_df = summary_knn_vote_experiment(
    knn_vote_results + [ml_result],
    output_path="loss_experiments/outputs",
    output_name="knn_vote_vs_ml_comparison.png",
    plot_variants=(*_KNN_VOTE_DEFAULT_VARIANTS, "ML classifier (XGBoost)"),
)

best_row = knn_vote_df.iloc[0]
best_knn = best_row["val_macro_f1"]
print(f"\n>> Best overall by macro-F1: {best_row['variant_name']}, macro-F1 ~ {best_knn:.3f}.")

knn_prior_all = all_lab.mean(1)   # reuse as a fusion prior if wanted

# Rich-token model

In [ ]:
def _soc_labelquery_factory():
    return SOCLabelQueryModel(
        token_dim=1052,
        hidden=768,                 # #4: smaller than the 768 baseline; bump to 512 if it underfits
        num_soc_classes=num_soc_classes,
        dropout=0.2,
        prior_logits=soc_prior_logits,
        num_heads=4,
    )

LOSS_CMP_EPOCHS = 50   # e.g. 30 for quick check; 50 for full comparison
LOSS_CMP_PATIENCE = 5
LOSS_CMP_SEEDS = (1234, 42, 7) 


In [ ]:
summary_comparison_experiment(loss_multihot_df,
    output_name="loss_multihot_f1_recall_precision_706.png",   
    plot=True,
    verbose=True,
    legend_loc="upper center"
    )

In [ ]:
# Create one checkpoint per seed from the trained rich-token runs and evaluate on the OOT loader.
seed_checkpoint_paths = {}
for seed in (1234, 42, 7):
    best_entry = None
    best_threshold = 0.3
    for loss_name in sorted(loss_multihot_artifacts.keys()):
        entry = loss_multihot_artifacts[loss_name].get(seed)
        if entry is None:
            continue
        result = entry.get("result")
        if result is None:
            continue
        val_f1 = getattr(result, "val_macro_f1", None)
        if val_f1 is None:
            continue
        if best_entry is None or val_f1 > best_entry[0]:
            best_entry = (val_f1, loss_name, entry)
            best_threshold = getattr(result, "threshold", 0.3)
    if best_entry is None:
        continue
    _, loss_name, entry = best_entry
    model = entry.get("model")
    if model is None or not isinstance(model, torch.nn.Module):
        model = _rich_labelquery_factory().to(device)
        model.load_state_dict(entry["model_state_dict"])
    else:
        model = model.to(device)
    model.eval()
    ckpt_path = Path(f"./loss_experiments/outputs/rich_token_seed_{seed}.pt")
    torch.save(model.state_dict(), ckpt_path)
    seed_checkpoint_paths[seed] = (ckpt_path, best_threshold)
    print(f"Saved {loss_name} checkpoint for seed {seed} -> {ckpt_path}")

oot_rows = []
for seed, (ckpt_path, threshold) in sorted(seed_checkpoint_paths.items()):
    model = _rich_labelquery_factory().to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    y_true, probs = collect_multihot_probs(model, rich_oot_loader)
    pred = (probs >= threshold).astype(float)
    prec_macro = precision_score(y_true, pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_true, pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, pred, average="macro", zero_division=0)
    prec_micro = precision_score(y_true, pred, average="micro", zero_division=0)
    rec_micro = recall_score(y_true, pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, pred, average="micro", zero_division=0)
    auprc_micro = average_precision_score(y_true, probs, average="micro")
    oot_rows.append({
        "seed": seed,
        "checkpoint": str(ckpt_path),
        "threshold": threshold,
        "macro_precision": prec_macro,
        "macro_recall": rec_macro,
        "macro_f1": f1_macro,
        "micro_precision": prec_micro,
        "micro_recall": rec_micro,
        "micro_f1": f1_micro,
        "micro_auprc": auprc_micro,
    })

oot_eval_df = pd.DataFrame(oot_rows).sort_values("macro_f1", ascending=False)
print("\nOOT evaluation for rich-token checkpoints:")
display(oot_eval_df)
oot_eval_df.to_csv("./loss_experiments/outputs/rich_token_oot_eval.csv", index=False)

In [ ]:
# ============================================================================
# # Higher-AUPRC SCORE-WEIGHTED all-10 prior; prior & model share native rows.
# Aggregated over seeds -> macro-F1 mean ± std (error bars).
# ============================================================================
import numpy as np, pandas as pd, importlib
import util.multihot_loss_comparison as my_module
importlib.reload(my_module)
from util.multihot_loss_comparison import *
# collect_multihot_probs is a notebook-defined helper (already in scope)

# compare 2 Loss Types by 3 grouped classes - freq/medium/rare


In [ ]:
importlib.reload(my_module)
from util.multihot_loss_comparison import *

pt_soc_all = pickle.load(open("../pt_soc_all.pkl", "rb"))
soc_classes = pt_soc_all["soc_name"].unique().tolist()

# Per-class F1 on validation (uses each run's tuned threshold)
loss_rich_per_class_long, loss_rich_per_class_wide, \
loss_rich_bin_summary, loss_rich_delta_by_bin = (
    run_per_class_f1_breakdown(
        artifacts=loss_multihot_artifacts,
        loader=rich_oot_loader,
        collect_multihot_probs=collect_multihot_probs,
        pos_rates=soc_pos_rate,
        class_names=soc_classes,
        baseline_loss="softmax",
        target_loss="zero_inflated",
        output_dir="experiments/outputs",
        plot=True,
        verbose=True,
    )
)

# Wide table: F1 per loss + ZeroInflated − BCE delta, sorted by rarity
display_cols = [
    "class_name", "pos_rate", "freq_bin", "bce", "focal", "zero_inflated",
    "delta_zero_inflated_minus_bce",
]
display_cols = [c for c in display_cols if c in loss_rich_per_class_wide.columns]
loss_rich_per_class_wide.sort_values("pos_rate")[display_cols]


In [ ]:
# # save soc_multihot score model
# torch.save(soc_multihot_model.state_dict(), "soc_multihot_score_model_603.pth")
# print("Saved SOC multi-hot model to soc_multihot_score_model_603.pth")


# Performance on Subclass Bins (rare, uncommon, common) — mean ± std over seeds

In [ ]:
# Per-class F1 on validation (uses each run's tuned threshold)
loss_multihot_per_class_long, loss_multihot_per_class_wide, \
loss_multihot_bin_summary, loss_multihot_delta_by_bin = (
    run_per_class_f1_breakdown(
        artifacts=loss_multihot_artifacts,
        loader=val_loader_1hot,
        collect_multihot_probs=collect_multihot_probs,
        pos_rates=soc_pos_rate,
        class_names=soc_classes,
        baseline_loss="softmax",
        target_loss="zero_inflated",
        output_dir="experiments/outputs",
        plot=True,
        verbose=True,
    )
)

# Wide table: F1 per loss + ZeroInflated − BCE delta, sorted by rarity
display_cols = [
    "class_name", "pos_rate", "freq_bin", "bce", "focal", "zero_inflated",
    "delta_zero_inflated_minus_bce",
]
display_cols = [c for c in display_cols if c in loss_multihot_per_class_wide.columns]
loss_multihot_per_class_wide.sort_values("pos_rate")[display_cols]

# Evaluate score only multi-hot model on 2025Q3-2026Q1

In [ ]:
# #load multihot w/o pt
# soc_multihot_model.load_state_dict(torch.load("soc_multihot_pt_model_603.pth", map_location=device))


# Inference on Q4'25-Q1'26 baselines: 
* soc_multihot_model
* soc_labelquery_model

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

soc_multihot_model.eval()
test_loss = 0.0
hamming = 0.0
exact_match = 0
jaccard_sum = 0.0
n_examples = 0
y_true_all = []
y_pred_all = []

with torch.no_grad():
    for x_batch, y_batch in oot_loader_1hot:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = soc_multihot_model(x_batch)
        loss = multihot_criterion(logits, y_batch)
        pred = (torch.sigmoid(logits) >= MULTIHOT_THRESHOLD).float()

        y_true_all.append(y_batch.cpu().numpy())
        y_pred_all.append(pred.cpu().numpy())

        bs = x_batch.size(0)
        test_loss += loss.item() * bs
        hamming += (pred != y_batch).float().mean(dim=1).sum().item()
        exact_match += (pred == y_batch).all(dim=1).sum().item()

        inter = (pred * y_batch).sum(dim=1)
        union = ((pred + y_batch) > 0).float().sum(dim=1)
        jaccard_sum += (inter / union.clamp(min=1e-8)).sum().item()
        n_examples += bs

y_true = np.vstack(y_true_all)
y_pred = np.vstack(y_pred_all)

test_loss /= n_examples
hamming /= n_examples
exact_match /= n_examples
mean_jaccard = jaccard_sum / n_examples

print(f"2025q2 test loss: {test_loss:.4f} | threshold={MULTIHOT_THRESHOLD:.2f}")
print(f"2025q2 pred label density: {y_pred.mean():.4f} (train density {y_train_mh.mean():.4f})")
print(f"2025q2 test Hamming loss: {hamming:.4f}")
print(f"2025q2 test subset accuracy: {exact_match:.4f}")
print(f"2025q2 test mean Jaccard: {mean_jaccard:.4f}")
print(f"2025q2 test cases: {n_examples}")


In [ ]:
from sklearn.metrics import average_precision_score

# Multilabel precision / recall / F1 (micro = globally across all labels; macro = unweighted mean per SOC)
for avg in ("micro","macro", "weighted"):
    prec = precision_score(y_true, y_pred, average=avg, zero_division=0)
    rec = recall_score(y_true, y_pred, average=avg, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=avg, zero_division=0)
    print(f"2025q2 {avg} avg — Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

micro_auprc = average_precision_score(y_true, y_prob, average="micro")
print(f"2025q2 micro avg — AUPRC: {micro_auprc:.4f}")

# Per-SOC class metrics (for inspection)
per_soc = pd.DataFrame(
    {
        "soc": soc_classes,
        "precision": precision_score(y_true, y_pred, average=None, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=None, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=None, zero_division=0),
        "support": y_true.sum(axis=0),
    }
).sort_values("support", ascending=False) #sort by support count
print("\nPer-SOC metrics (top 10 by support):")
display(per_soc)


# Multilabel precision / recall / F1 
* micro = calculate metrics globally 
* macro = unweighted mean per SOC 
* weighted = by class support
* jaccard score = 
* hamming loss = 


# compare 10 neighbors from hybrid retrieval pipeline
* precision, recall, f1

In [ ]:
# Compare BM25 vs Dense Vector at rank 1 to 5 on 2025q2 test set (individual ranks, NO aggregation)
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

test_mask = oot["yr_qtr"] == "2025q2"
test_indices = np.where(test_mask.values)[0]
y_test = y_1hot[test_mask.values].numpy().astype(float)

print(f"\n{'='*100}")
print(f"2025Q2 Test Set: Individual Rank SOC Predictions (rank 1 to 5, NO aggregation)")
print(f"Test cases: {len(test_indices)} | Ground truth avg labels: {y_test.mean(axis=1).mean():.2f}")
print(f"{'='*100}\n")

# For each rank position, extract and compare BM25 vs Dense
results_all = []

for rank in range(1, 6):
    #print(f"\n--- Rank {rank} (index {rank-1}) ---")
    
    # BM25: take only rank-th retrieval (index rank-1)
    y_bm25_rank = np.zeros_like(y_test)
    for idx_pos, case_idx in enumerate(test_indices):
        case_data = np.asarray(bm25_soc_1hot[case_idx], dtype=np.float32)  # shape: (5, n_soc)
        if case_data.shape[0] >= rank:
            # Take only the rank-th row (index rank-1)
            y_bm25_rank[idx_pos] = case_data[rank-1]
    
    # Dense Vector (VECT): take only rank-th retrieval (index rank-1)
    y_vect_rank = np.zeros_like(y_test)
    for idx_pos, case_idx in enumerate(test_indices):
        case_data = np.asarray(vect_soc_1hot[case_idx], dtype=np.float32)  # shape: (5, n_soc)
        if case_data.shape[0] >= rank:
            # Take only the rank-th row (index rank-1)
            y_vect_rank[idx_pos] = case_data[rank-1]
    
    # Compute metrics for each method at this rank
    methods_rank = {
        "BM25": y_bm25_rank,
        "Dense": y_vect_rank,
    }
    
    for method_name, y_pred in methods_rank.items():
        # Micro-averaged
        prec_micro = precision_score(y_test, y_pred, average='micro', zero_division=0)
        rec_micro = recall_score(y_test, y_pred, average='micro', zero_division=0)
        f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
        
        # Macro-averaged
        prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        
        # Weighted-averaged
        prec_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        results_all.append({
            'Rank': rank,
            'Method': method_name,
            'P_Micro': prec_micro,
            'R_Micro': rec_micro,
            'F1_Micro': f1_micro,
            'P_Macro': prec_macro,
            'R_Macro': rec_macro,
            'F1_Macro': f1_macro,
            'P_Weighted': prec_weighted,
            'R_Weighted': rec_weighted,
            'F1_Weighted': f1_weighted,
        })
        
        # print(f"  {method_name:10} | Micro: P={prec_micro:.4f} R={rec_micro:.4f} F1={f1_micro:.4f} "
        #       f"| Macro: P={prec_macro:.4f} R={rec_macro:.4f} F1={f1_macro:.4f} "
        #       f"| Weighted: P={prec_weighted:.4f} R={rec_weighted:.4f} F1={f1_weighted:.4f}")

# Create full table
df_full = pd.DataFrame(results_all)

# Group by Method
# print(f"\n\n{'='*100}")
# print("GROUPED BY METHOD: Individual Rank Comparison (2025Q2 OOT Test Set)")
# print(f"{'='*100}\n")

for method in ['BM25', 'Dense']:
    print(f"\n{'─'*100}")
    print(f"{method}")
    print(f"{'─'*100}")
    df_method = df_full[df_full['Method'] == method][['Rank', 'P_Micro', 'R_Micro', 'F1_Micro', 
                                                         'P_Macro', 'R_Macro', 'F1_Macro',
                                                         'P_Weighted', 'R_Weighted', 'F1_Weighted']]
    display(df_method.set_index('Rank'))

print(f"\n\n{'='*100}")
print("FULL TABLE (for reference)")
print(f"{'='*100}\n")
display(df_full.rename(columns={"Rank": "TopK"}).sort_values(['Method', 'TopK']).reset_index(drop=True))


# Compare benchmark DL models
* LSTM and BiLSTM baselines for SOC multi-hot comparison


In [ ]:
# 10 X (1+27+1024) tensor per case
x_1hot_tensor.shape

In [ ]:
class SOCLSTMMultiHotModel(nn.Module):
    """LSTM baseline: input (B, 10, 1052) -> logits (B, 27)."""
    def __init__(self, token_dim=1052, hidden=1024, num_soc_classes=27, 
                 dropout=0.2, prior_logits=None):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=token_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
            dropout=0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_soc_classes),
        )
        if prior_logits is not None:
            with torch.no_grad():
                self.head[-1].bias.copy_(prior_logits.to(self.head[-1].bias.device))

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 10, -1)
        _, (h_n, _) = self.lstm(x)
        h_last = h_n[-1]  # (B, hidden)
        return self.head(h_last)


class SOCBiLSTMMultiHotModel(nn.Module):
    """BiLSTM baseline: input (B, 10, 1052) -> logits (B, 27)."""
    def __init__(self, token_dim=1052, hidden=1024, num_soc_classes=27, dropout=0.2, prior_logits=None):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=token_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_soc_classes),
        )
        if prior_logits is not None:
            with torch.no_grad():
                self.head[-1].bias.copy_(prior_logits.to(self.head[-1].bias.device))

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 10, -1)
        _, (h_n, _) = self.lstm(x)
        # bidirectional: concatenate last layer's forward/backward hidden states
        h_cat = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (B, 2*hidden)
        return self.head(h_cat)


# --- LSTM and BiLSTM with Attention mechanisms ---

class SOCLSTMMultiHotModelWithAttention(nn.Module):
    """LSTM with multi-head attention: input (B, 10, 1052) -> logits (B, 27)."""
    def __init__(self, token_dim=1052, hidden=1024, num_soc_classes=27, 
                 dropout=0.2, prior_logits=None, num_heads=4):
        super().__init__()
        self.hidden = hidden
        self.lstm = nn.LSTM(
            input_size=token_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
            dropout=0.0,
        )
        # Multi-head attention over sequence
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.attention_norm = nn.LayerNorm(hidden)
        
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_soc_classes),
        )
        if prior_logits is not None:
            with torch.no_grad():
                self.head[-1].bias.copy_(prior_logits.to(self.head[-1].bias.device))

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 10, -1)
        # LSTM forward: (B, 10, 1052) -> (B, 10, hidden)
        lstm_out, _ = self.lstm(x)
        
        # Multi-head attention: (B, 10, hidden) -> (B, 10, hidden)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Residual connection and layer norm
        lstm_out = self.attention_norm(lstm_out + attn_out)
        
        # Global average pooling over sequence
        pooled = lstm_out.mean(dim=1)  # (B, hidden)
        
        return self.head(pooled)


class SOCBiLSTMMultiHotModelWithAttention(nn.Module):
    """BiLSTM with multi-head attention: input (B, 10, 1052) -> logits (B, 27)."""
    def __init__(self, token_dim=1052, hidden=1024, num_soc_classes=27, 
                 dropout=0.2, prior_logits=None, num_heads=4):
        super().__init__()
        self.hidden = hidden
        self.lstm = nn.LSTM(
            input_size=token_dim,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0.0,
        )
        # Multi-head attention over sequence (bidirectional output size = hidden*2)
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden * 2,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.attention_norm = nn.LayerNorm(hidden * 2)
        
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_soc_classes),
        )
        if prior_logits is not None:
            with torch.no_grad():
                self.head[-1].bias.copy_(prior_logits.to(self.head[-1].bias.device))

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 10, -1)
        # BiLSTM forward: (B, 10, 1052) -> (B, 10, hidden*2)
        lstm_out, _ = self.lstm(x)
        
        # Multi-head attention: (B, 10, hidden*2) -> (B, 10, hidden*2)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Residual connection and layer norm
        lstm_out = self.attention_norm(lstm_out + attn_out)
        
        # Global average pooling over sequence
        pooled = lstm_out.mean(dim=1)  # (B, hidden*2)
        
        return self.head(pooled)


# Compare benchmark CNN models

In [ ]:
# --- CNN models without and with attention ---
class SOCCNNMultiHotModel(nn.Module):
    """CNN baseline: input (B, 10, 1052) -> logits (B, 27).
    Uses 1D convolutions with multiple kernel sizes for multi-scale feature extraction.
    """
    def __init__(self, token_dim=1052, hidden=1024, num_soc_classes=27,
                 dropout=0.2, prior_logits=None, num_filters=64, kernel_sizes=(3, 5, 7)):
        super().__init__()
        self.hidden = hidden
        self.num_filters = num_filters
        self.kernel_sizes = kernel_sizes

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=token_dim, out_channels=num_filters,
                      kernel_size=ks, padding=ks // 2)
            for ks in kernel_sizes
        ])

        self.bn_layers = nn.ModuleList([
            nn.BatchNorm1d(num_filters) for _ in kernel_sizes
        ])

        cnn_output_dim = num_filters * len(kernel_sizes)

        self.head = nn.Sequential(
            nn.Linear(cnn_output_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_soc_classes),
        )

        if prior_logits is not None:
            with torch.no_grad():
                self.head[-1].bias.copy_(prior_logits.to(self.head[-1].bias.device))

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 10, -1)

        x = x.transpose(1, 2)  # (B, token_dim, seq_len)

        conv_outputs = []
        for conv, bn in zip(self.convs, self.bn_layers):
            conv_out = conv(x)
            conv_out = bn(conv_out)
            conv_out = F.relu(conv_out)
            pooled = conv_out.mean(dim=2)
            conv_outputs.append(pooled)

        pooled = torch.cat(conv_outputs, dim=1)
        return self.head(pooled)

class SOCCNNMultiHotModelWithAttention(nn.Module):
    """CNN + self-attention refinement (apples-to-apples vs baseline).

    Key difference vs `SOCCNNMultiHotModel`: after the CNN stack, we run a
    self-attention block over the sequence, but we keep the *same mean pooling*
    over sequence length for the final representation.
    """

    def __init__(self, token_dim=1052, hidden=1024, num_soc_classes=27,
                 dropout=0.2, prior_logits=None, num_filters=64, kernel_sizes=(3, 5, 7),
                 num_heads=4):
        super().__init__()
        self.hidden = hidden
        self.num_filters = num_filters
        self.kernel_sizes = kernel_sizes

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=token_dim, out_channels=num_filters,
                      kernel_size=ks, padding=ks // 2)
            for ks in kernel_sizes
        ])

        self.bn_layers = nn.ModuleList([
            nn.BatchNorm1d(num_filters) for _ in kernel_sizes
        ])

        cnn_output_dim = num_filters * len(kernel_sizes)
        self.cnn_proj = nn.Linear(cnn_output_dim, hidden)

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attention_norm = nn.LayerNorm(hidden)

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_soc_classes),
        )

        if prior_logits is not None:
            with torch.no_grad():
                self.head[-1].bias.copy_(prior_logits.to(self.head[-1].bias.device))

    def forward(self, x):
        if x.dim() == 2:
            x = x.view(x.size(0), 10, -1)

        x_conv = x.transpose(1, 2)  # (B, token_dim, seq_len)

        conv_outputs = []
        for conv, bn in zip(self.convs, self.bn_layers):
            conv_out = conv(x_conv)
            conv_out = bn(conv_out)
            conv_out = F.relu(conv_out)
            conv_outputs.append(conv_out)

        concat_conv = torch.cat(conv_outputs, dim=1)  # (B, C, seq_len)
        concat_conv = concat_conv.transpose(1, 2)     # (B, seq_len, C)

        cnn_seq = self.cnn_proj(concat_conv)          # (B, seq_len, hidden)

        attn_out, _ = self.attention(cnn_seq, cnn_seq, cnn_seq)
        cnn_seq = self.attention_norm(cnn_seq + attn_out)

        pooled = cnn_seq.mean(dim=1)                  # (B, hidden)
        return self.head(pooled)


In [ ]:
def train_single_multihot_model(
    model,
    train_loader,
    val_loader,
    oot_loader,
    criterion,
    epochs=50,
    lr=1e-4,
    weight_decay=1e-4,
    threshold=0.27, # precission floor value 
    tag="model",
    patience=5,
    random_state=None,
):
    if random_state is not None:
        torch.manual_seed(random_state)
        torch.cuda.manual_seed_all(random_state)
        np.random.seed(random_state)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state, best_val_f1 = None, -1.0
    epochs_no_improve = 0

    for ep in range(1, epochs + 1):
        model.train()
        tr_loss, n = 0.0, 0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device).float()

            # e.g. logits = soc_multihot_model(x_batch)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            bs = x_batch.size(0)
            tr_loss += loss.item() * bs
            n += bs

        tr_loss /= max(n, 1)
        va_loss, va_f1, va_rec, va_pred_density = eval_multihot_loader(
            model, val_loader, criterion, threshold=threshold
        )

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(
            f"[{tag}] ep {ep:02d}/{epochs} | "
            f"tr_loss={tr_loss:.4f} va_loss={va_loss:.4f} "
            f"va_f1={va_f1:.4f} va_rec={va_rec:.4f} pred_density={va_pred_density:.4f} "
            f"(patience: {epochs_no_improve}/{patience})"
        )

        if epochs_no_improve >= patience:
            print(f"[{tag}] Early stopping at epoch {ep} (no improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_oot, oot_probs = collect_multihot_probs(model, oot_loader)
    # enforce precision floor when tuning threshold (use `threshold` as precision floor)
    oot_threshold, f1 = tune_multihot_threshold_f1(y_oot, oot_probs)

    y_pred = (oot_probs >= oot_threshold).astype(float)
    final_p = precision_score(y_oot, y_pred, average="macro", zero_division=0)
    final_r = recall_score(y_oot, y_pred, average="macro", zero_division=0)
    final_f1 = f1_score(y_oot, y_pred, average="macro", zero_division=0)
    final_hl = hamming_loss(y_oot, y_pred)
    final_jc = jaccard_score(y_oot, y_pred, average="samples", zero_division=0)
    final_auc = roc_auc_score(y_oot, y_pred, average="macro")
    final_pr_auc = average_precision_score(y_oot, y_pred, average="macro")
    density = y_pred.mean()

    return {
        "model": model,        
        "val_precision": float(final_p),
        "val_recall": float(final_r),
        "val_f1": float(final_f1),
        "val_auc": float(final_auc),
        "val_prc":float(final_pr_auc),
        "val_hl":float(final_hl),
        "val_jc":float(final_jc),
        "density":float(density)
    }


In [ ]:
TOKEN_DIM = 1052  # should be 1052 based on earlier print
num_soc_classes = 27
criterion =  nn.BCEWithLogitsLoss().to(device)

SEEDS = [1234, 24, 7]


def build_model_zoo():
    # --- Build models for direct comparison with SOCMultiHotModel() ---
    attn_model = SOCLabelQueryModel(
        token_dim=TOKEN_DIM,
        hidden=768,
        num_soc_classes=num_soc_classes,
        dropout=0.2,
        prior_logits=rich_prior_logits,
        num_heads=4,
    ).to(device)

    lstm_model = SOCLSTMMultiHotModel(
        token_dim=TOKEN_DIM,
        hidden=768,
        num_soc_classes=num_soc_classes,
        dropout=0.2,
        prior_logits=rich_prior_logits,
    ).to(device)

    bilstm_model = SOCBiLSTMMultiHotModel(
        token_dim=TOKEN_DIM,
        hidden=768,
        num_soc_classes=num_soc_classes,
        dropout=0.2,
        prior_logits=rich_prior_logits,
    ).to(device)

    cnn_model = SOCCNNMultiHotModel(
        token_dim=TOKEN_DIM,
        hidden=768,
        num_soc_classes=num_soc_classes,
        dropout=0.2,
        prior_logits=rich_prior_logits,
        num_filters=64,
        kernel_sizes=(3, 5, 7)
    ).to(device)

    return {
        "SOCMultiHotModel": attn_model,
        "LSTM_baseline": lstm_model,
        "BiLSTM_baseline": bilstm_model,
        "CNN_Baseline": cnn_model,
    }


In [ ]:
# Train all models across 3 seeds (fresh weight init per seed) for mean/std comparison
metric_cols = ["val_precision", "val_recall", "val_f1", "val_auc", "val_prc", "val_hl", "val_jc", "density"]
comparison_rows = []
for seed in tqdm(SEEDS):
    torch.manual_seed(seed)  # re-init weights differently per seed before building
    model_zoo = build_model_zoo()
    for name, model in model_zoo.items():
        result = train_single_multihot_model(
            model,
            rich_train_loader,
            rich_val_loader,
            rich_oot_loader,
            criterion=criterion,
            epochs=50,
            lr=1e-4,
            weight_decay=1e-4,
            threshold=0.27, # precission floor
            tag=f"{name}_seed{seed}",
            random_state=seed,
        )
        comparison_rows.append({"model": name, "seed": seed, **{c: result[c] for c in metric_cols}})

results_df = pd.DataFrame(comparison_rows)
summary_df = (
    results_df.groupby("model")[metric_cols]
    .agg(["mean", "std"])
    .sort_values(("val_f1", "mean"), ascending=False)
)
print("\nValidation comparison across 3 seeds (mean / std):")
display(summary_df)

# save evaluation on DL models on OOT set

In [ ]:
results_df.to_csv("dl_model_comparison_oot_719.csv", index=False)

In [ ]:
def train_label_query_model(seeds=(1234, 24, 7), epochs=50, lr=1e-4, weight_decay=1e-4, threshold=0.27, patience=5):
    """Train the Label-Query Attention model (SOCLabelQueryModel) across 3 seeds on the
    rich evidence, then report:
      Part 1 -- per-class F1 on the OOT split, binned into rare/medium/frequent SOC
                tertiles by training-set prevalence, mean +/- std over seeds.
      Part 2 -- a qualitative example per seed: one OOT case, its 10 retrieved
                neighbors (own SOC labels + retrieval score), and the model's
                per-SOC attention weight over those neighbors for its top prediction.
    """
    criterion = nn.BCEWithLogitsLoss().to(device)
    train_prevalence = (y_tr.sum(0) / y_tr.shape[0]).cpu().numpy()

    trained_models = []
    per_seed_tertiles = []
    qualitative_examples = []

    for seed in tqdm(seeds):
        torch.manual_seed(seed)
        model = _rich_labelquery_factory().to(device)

        result = train_single_multihot_model(
            model,
            rich_train_loader,
            rich_val_loader,
            rich_oot_loader,
            criterion=criterion,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            threshold=threshold,
            tag=f"LabelQueryAttention_seed{seed}",
            patience=patience,
            random_state=seed,
        )
        model = result["model"]
        trained_models.append(model)

        # --- run it with the seed-1234 model already in `results["models"]` ---
        #model = result["models"][0]
        y_oot, oot_probs = collect_multihot_probs(model, rich_oot_loader)
        oot_threshold, _ = tune_multihot_threshold_f1(y_oot, oot_probs)

        best = find_good_qualitative_example(model, rich_oot_loader, oot_threshold)
        print(f"Best case: exact_match={best['exact_match']}  jaccard={best['jaccard']:.3f}")
        print(f"  True SOCs:      {best['true_socs']}")
        print(f"  Predicted SOCs: {best['predicted_socs']}")

        # --- Part 1: per-class P/R/F1 by SOC frequency tertile, this seed's OOT predictions ---
        model.eval()
        y_oot, oot_probs = collect_multihot_probs(model, rich_oot_loader)
        oot_threshold, _ = tune_multihot_threshold_f1(y_oot, oot_probs)
        y_pred = (oot_probs >= oot_threshold).astype(float)
        per_class_p = precision_score(y_oot, y_pred, average=None, zero_division=0)
        per_class_r = recall_score(y_oot, y_pred, average=None, zero_division=0)
        per_class_f1 = f1_score(y_oot, y_pred, average=None, zero_division=0)

        freq_bin_df = pd.DataFrame({
            "soc": SOC_NAMES,
            "train_prevalence": train_prevalence,
            "precision": per_class_p,
            "recall": per_class_r,
            "f1": per_class_f1,
        })
        freq_bin_df["tertile"] = pd.qcut(freq_bin_df["train_prevalence"], 3, labels=["Rare", "Medium", "Frequent"])
        freq_bin_df["seed"] = seed
        per_seed_tertiles.append(freq_bin_df)

        # --- Part 2: qualitative example (retrieved neighbors + attention), this seed ---
        x_example = best["x_example"].to(device)
        with torch.no_grad():
            logits, attn_w = model(x_example, return_attn=True)
            probs = torch.sigmoid(logits)[0].cpu().numpy()

        j = SOC_NAMES.index(best["predicted_socs"][0])
        alpha = attn_w[0, j].detach().cpu().numpy()
        tokens = x_example[0].cpu().numpy()

        neighbor_labels = [f"BM25 Top{i+1}" for i in range(5)] + [f"Dense Top{i+1}" for i in range(5)]
        rows = [
            {
                "neighbor": neighbor_labels[i],
                "retrieval_score": float(tok[0]),
                "attention_weight": float(alpha[i]),
                "shares_predicted_label": best["predicted_socs"][0] in [SOC_NAMES[k] for k, v in enumerate(tok[1:28]) if v],
            }
            for i, tok in enumerate(tokens)
        ]
        example_df = pd.DataFrame(rows).sort_values("attention_weight", ascending=False)
        display(example_df)        

    # --- aggregate Part 1 across seeds: per-class table + tertile summary ---
    all_tertiles = pd.concat(per_seed_tertiles, ignore_index=True)

    per_class_results = (
        all_tertiles.groupby("soc")[["precision", "recall", "f1"]]
        .agg(["mean", "std"])
    )
    per_class_results.columns = ["_".join(c) for c in per_class_results.columns]
    per_class_results = per_class_results.reset_index().merge(
        all_tertiles[["soc", "train_prevalence", "tertile"]].drop_duplicates(), on="soc"
    ).sort_values("train_prevalence")
    per_class_results.to_csv("per_class_results_719.csv", index=False)

    tertile_summary = (
        all_tertiles.groupby("tertile")["f1"]
        .agg(["mean", "std"])
        .reindex(["Rare", "Medium", "Frequent"])
    )
    print("\nPer-class results (27 SOCs, mean +/- std over 3 seeds), saved to per_class_results_719.csv:")
    display(per_class_results)
    print("\nPer-class F1 by SOC frequency tertile, mean +/- std over 3 seeds (OOT split):")
    display(tertile_summary)

    return {
        "models": trained_models,
        "per_class_results": per_class_results,
        "tertile_summary": tertile_summary,
        "per_seed_tertiles": all_tertiles,
        "qualitative_examples": qualitative_examples,
    }


results = train_label_query_model(seeds=[1234, 24, 7])
per_class_results = results["per_class_results"]

In [ ]:
pickle.dump(results, open("per_class_results.pkl", "wb"))
#per_class_results = pickle.load(open("per_class_results.pkl", "rb"))

In [ ]:
results.keys()

In [ ]:
def find_good_qualitative_example(model, loader, oot_threshold, 
                                  min_true_socs=2, max_true_socs=6):
    """Scan the whole OOT loader for the case with the best agreement between true and
    predicted SOC sets (exact match preferred, then highest Jaccard), among cases with at
    least `min_true_socs` true labels (avoids a trivial single-label case as the example).
    """
    model.eval()
    best = None
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch.to(device))
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= oot_threshold).astype(int)
            y_np = y_batch.numpy()
            for i in range(len(y_np)):
                true_idx = [j for j, v in enumerate(y_np[i]) if v]
                pred_idx = [j for j, v in enumerate(preds[i]) if v]
                if len(true_idx) < min_true_socs or len(true_idx) > max_true_socs:
                    continue
                true_set, pred_set = set(true_idx), set(pred_idx)
                union = true_set | pred_set
                jaccard = len(true_set & pred_set) / len(union) if union else 0.0
                exact = true_set == pred_set
                score = (exact, jaccard, len(true_set))
                if best is None or score > best["score"]:
                    best = {
                        "score": score,
                        "x_example": x_batch[i : i + 1].clone(),
                        "y_example": y_batch[i].clone(),
                        "true_socs": [SOC_NAMES[j] for j in true_idx],
                        "predicted_socs": [SOC_NAMES[j] for j in pred_idx],
                        "jaccard": jaccard,
                        "exact_match": exact,
                    }
    return best

# Explain SOC prediction using attention weights learned by the model

In [ ]:
def find_joint_qualitative_example(models, loader, oot_thresholds, 
                                   min_true_socs=2, max_true_socs=6):
    """Scan the whole OOT loader for one case where ALL of `models` (one per seed) agree well
    with the true SOC set (all-exact-match preferred, then highest worst-case Jaccard across
    models), among cases with a realistic label count. Returns the shared case plus each
    model's own predictions on it.
    """
    for m in models:
        m.eval()
    best = None
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_dev = x_batch.to(device)
            y_np = y_batch.numpy()
            all_preds = [
                (torch.sigmoid(model(x_dev)).cpu().numpy() >= thr).astype(int)
                for model, thr in zip(models, oot_thresholds)
            ]

            for i in range(len(y_np)):
                true_idx = [j for j, v in enumerate(y_np[i]) if v]
                if len(true_idx) < min_true_socs or len(true_idx) > max_true_socs:
                    continue
                true_set = set(true_idx)

                jaccards, exacts, pred_socs_per_model = [], [], []
                for preds in all_preds:
                    pred_idx = [j for j, v in enumerate(preds[i]) if v]
                    pred_set = set(pred_idx)
                    union = true_set | pred_set
                    jaccards.append(len(true_set & pred_set) / len(union) if union else 0.0)
                    exacts.append(true_set == pred_set)
                    pred_socs_per_model.append([SOC_NAMES[j] for j in pred_idx])

                score = (all(exacts), min(jaccards), len(true_set))
                if best is None or score > best["score"]:
                    best = {
                        "score": score,
                        "x_example": x_batch[i : i + 1].clone(),
                        "true_socs": [SOC_NAMES[j] for j in true_idx],
                        "per_model_predicted_socs": pred_socs_per_model,
                        "per_model_jaccard": jaccards,
                        "all_exact_match": all(exacts),
                    }
    return best

In [ ]:
seeds = [1234, 24, 7]
models = results['models']

oot_thresholds = []
for model in tqdm(models):
    y_oot, oot_probs = collect_multihot_probs(model, rich_oot_loader)
    thr, _ = tune_multihot_threshold_f1(y_oot, oot_probs)
    oot_thresholds.append(thr)

best = find_joint_qualitative_example(models, rich_oot_loader, oot_thresholds)
print(f"All-exact-match across seeds: {best['all_exact_match']}  per-seed Jaccard: {best['per_model_jaccard']}")
print(f"True SOCs: {best['true_socs']}")
for seed, pred in zip(seeds, best["per_model_predicted_socs"]):
    print(f"  seed {seed} predicted: {pred}")

# target SOC: one all 3 models correctly predicted (a genuine shared true positive)
common_pred = set(best["true_socs"])
for pred in best["per_model_predicted_socs"]:
    common_pred &= set(pred)
target_soc = sorted(common_pred)[0] if common_pred else best["true_socs"][0]
j = SOC_NAMES.index(target_soc)

x_example = best["x_example"].to(device)
tokens = x_example[0].cpu().numpy()
neighbor_labels = [f"BM25 Top{i+1}" for i in range(5)] + [f"Dense Top{i+1}" for i in range(5)]

alphas = []
with torch.no_grad():
    for model in models:
        _, attn_w = model(x_example, return_attn=True)
        alphas.append(attn_w[0, j].detach().cpu().numpy())
alphas = np.stack(alphas)  # (3 seeds, 10 neighbors) -- retrieval is deterministic, only attention varies by seed

rows = [
    {
        "neighbor": neighbor_labels[i],
        "retrieval_score": float(tokens[i][0]),
        "attention_weight_mean": float(alphas[:, i].mean()),
        "attention_weight_std": float(alphas[:, i].std()),
        "shares_predicted_label": target_soc in [SOC_NAMES[k] for k, v in enumerate(tokens[i][1:28]) if v],
    }
    for i in range(len(tokens))
]
example_df = pd.DataFrame(rows).sort_values("attention_weight_mean", ascending=False)
print(f"\nTarget SOC (shared true positive across all 3 seeds): {target_soc}")
display(example_df)

In [ ]:
pickle.dump(example_df, open("cross_seed_attention.pkl", "wb")) 